# EECE 5644 Mini Project 1

In [ ]:
############################################################

# EECE 5644 - Machine Learning
# Skylar Denno
# June 30, 2026

# MINI PROJECT 1
# Data cleaning and data preprocessing

############################################################

In [ ]:
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
np.random.seed(0)

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

In [ ]:
class BasicDescription:
    def __init__(self, df):
        self.df = df

    def print_basic_description(self):
        print("\n\n--------------------\nBASIC DESCRIPTION\n--------------------")
        print("\nLoaded dataset with", self.df.shape[0], "rows")
        print("\nShape:", self.df.shape)
        print("\nColumns:", list(self.df.columns))
        print("\nInfo:", self.df.info())
        print("\nStatistical Description:", self.df.describe())
        print("\nHead (first 10 rows):", self.df.head(10))

basic_info = BasicDescription(raw_df)
basic_info.print_basic_description()

In [ ]:
class UniqueValues:
    def __init__(self, df):
        self.df = df

    def print_unique_values(self):
        print("\n\n--------------------\nUNIQUE VALUES/COUNTS\n--------------------")
        print("\nNum. of instances of each country present: ", self.df["Country"].value_counts())
        print("\nNum. of unique Stock Codes: ", self.df["StockCode"].nunique())
        print("\nNum. of unique product descriptions:", self.df["Description"].nunique())

unique_values = UniqueValues(raw_df)
unique_values.print_unique_values()

In [ ]:
class NumericSummary:
    def __init__(self, df):
        self.df = df

    def print_numeric_summary(self):
        print("\n\n--------------------\nNUMERIC SUMMARY\n--------------------")
        summary = self.df[["Quantity", "UnitPrice"]].agg(["min", "max", "sum", "mean", "count"])
        print(summary)

numeric_summary = NumericSummary(raw_df)
numeric_summary.print_numeric_summary()

In [ ]:
class CountryRegionLookup:
    def __init__(self):
        self.country_region_lookup = None

    def build_country_region_lookup(self):
        country_region_data = [
            ("United Kingdom", "UK & IE"),
            ("Germany", "Western Europe"),
            ("France", "Western Europe"),
            ("Spain", "Western Europe"),
            ("Netherlands", "Western Europe"),
            ("Belgium", "Western Europe"),
            ("Switzerland", "Western Europe"),
            ("Portugal", "Western Europe"),
            ("Australia", "Oceania"),
            ("Norway", "Northern Europe"),
            ("Italy", "Southern Europe"),
            ("Channel Islands", "Western Europe"),
            ("Finland", "Northern Europe"),
            ("Cyprus", "Mediterranean"),
            ("Sweden", "Northern Europe"),
            ("Austria", "Western Europe"),
            ("Denmark", "Northern Europe"),
            ("Japan", "Asia Pacific"),
            ("Poland", "Central Europe"),
            ("Israel", "Middle East"),
            ("USA", "North America"),
            ("Hong Kong", "Asia Pacific"),
            ("Singapore", "Asia Pacific"),
            ("Iceland", "Northern Europe"),
            ("Canada", "North America"),
            ("Greece", "Southern Europe"),
            ("Malta", "Southern Europe"),
            ("United Arab Emirates", "Middle East"),
            ("Lebanon", "Middle East"),
            ("Lithuania", "Eastern Europe"),
            ("Brazil", "South America"),
            ("Czech Republic", "Central Europe"),
            ("Bahrain", "Middle East"),
            ("Saudi Arabia", "Middle East"),
            ("RSA", "Africa"),
            ("ERIE", "Ireland"),
            ("European Community", "GENERAL EUROPE"),
            ("Unspecified", "UNSPECIFIED")
        ]

        country_region_dict = dict(country_region_data)
        country_region_dict.setdefault("DEFAULT", "INVALID COUNTRY")
        self.country_region_lookup = country_region_dict
        return self.country_region_lookup

country_lookup = CountryRegionLookup()
country_region_lookup_table = country_lookup.build_country_region_lookup()

In [ ]:
class SliceRows:
    def __init__(self, df):
        self.df = df

    def slice_rows(self, df=None):
        if df is None:
            df = self.df
        df = df.copy()
        df.index = [f"item_{i}" for i in range(len(df))]

        print("\n\n--------------------\nSLICING WITH ILOC AND LOC\n--------------------")
        print("\nFirst 5 rows with iloc:")
        print(df.iloc[0:5])

        labels = [f"item_{i}" for i in range(5)]
        print("\nRows retrieved by label with loc:")
        print(df.loc[labels])
        return df

slicer = SliceRows(clean_df)
clean_df = slicer.slice_rows(clean_df)

In [ ]:
class RemoveCanceledOrders:
    def __init__(self, df):
        self.df = df

    def remove_canceled_orders(self, df=None):
        if df is None:
            df = self.df
        df = df.copy()
        df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
        return df

canceled_remover = RemoveCanceledOrders(clean_df)
clean_df = canceled_remover.remove_canceled_orders(clean_df)

In [ ]:
class LargestOrders:
    def __init__(self, df):
        self.df = df

    def find_largest_orders(self, df=None, top_n=5):
        if df is None:
            df = self.df
        df = df.copy()
        df["LineRevenue"] = df["Quantity"] * df["UnitPrice"]

        print("\n\n--------------------\nTOP BULK ORDERS BY QUANTITY\n--------------------")
        print(df.sort_values("Quantity", ascending=False).head(top_n)[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "LineRevenue"]])

        print("\n\n--------------------\nTOP ORDERS BY LINE REVENUE\n--------------------")
        print(df.sort_values("LineRevenue", ascending=False).head(top_n)[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "LineRevenue"]])

        print("\n\n--------------------\nTOP RETURNS BY NEGATIVE QUANTITY\n--------------------")
        returns = df[df["Quantity"] < 0]
        print(returns.sort_values(["Quantity", "LineRevenue"], ascending=[True, True]).head(top_n)[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "LineRevenue"]])

        return df.sort_values(["Quantity", "LineRevenue"], ascending=[False, False])

largest_orders = LargestOrders(clean_df)
clean_df = largest_orders.find_largest_orders(clean_df)

In [ ]:
basic_info = BasicDescription(raw_df)
basic_info.print_basic_description()

unique_values = UniqueValues(raw_df)
unique_values.print_unique_values()

numeric_summary = NumericSummary(raw_df)
numeric_summary.print_numeric_summary()

country_lookup = CountryRegionLookup()
country_region_lookup_table = country_lookup.build_country_region_lookup()

slicer = SliceRows(clean_df)
clean_df = slicer.slice_rows(clean_df)

canceled_remover = RemoveCanceledOrders(clean_df)
clean_df = canceled_remover.remove_canceled_orders(clean_df)

largest_orders = LargestOrders(clean_df)
clean_df = largest_orders.find_largest_orders(clean_df)

cleaned_info = BasicDescription(clean_df)
cleaned_info.print_basic_description()